# Preprocess

In [1]:
import torch
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import gpytorch
import xarray as xr
import matplotlib.pyplot as plt

In [2]:
# Do this only to replicate
"""
# Load dataset
bed = xr.open_dataset("BedMachineAntarctica-v3.nc")

# y, x order, slice based on index, reduce to 28 MB
# keep all variables
bed_slice = bed.isel(y = slice(6666, 8666), x = slice(6000, 7400))

slice_filename = './BedMachineAntarctica-v3-slice.nc'
bed_slice.to_netcdf(path = slice_filename)
bed_slice.close()
print ('finished saving')
"""

In [11]:
bed = xr.open_dataset("BedMachineAntarctica-v3-slice.nc")

In [25]:
# 3 M values
bed.bed.values
np.max(bed.bed.values)

# Corners
bed.bed.sel(y = 0) # top left value is 68
bed.bed.sel(y = -999500) # bottom left value is -682
bed.bed.sel(x = -333000) # top left again
bed.bed.sel(x = 366500) # top right is -228

# South Pole
bed.bed.sel(y = 0, x = 0) # -27 at origin

<xarray.DataArray 'bed' ()>
array(-27.430664, dtype=float32)
Coordinates:
    x        int32 0
    y        int32 0
Attributes:
    long_name:      bed topography
    standard_name:  bedrock_altitude
    units:          meters
    grid_mapping:   mapping
    source:         IBCSO v2 and Mathieu Morlighem

In [26]:
# Magnify data because it is too large to visualise
magnification_factor = 100
magnify = torch.nn.AvgPool2d(kernel_size = magnification_factor)
# stride is my default the kernel size

input = torch.tensor(bed.bed.values).unsqueeze(0)
input = input.type(torch.DoubleTensor)
bed_magnified = magnify(input)
bed_magnified.shape

torch.Size([1, 20, 14])

In [31]:
fig = px.imshow(bed_magnified.squeeze(), 
                color_continuous_scale = 'RdBu_r',
                # DEFAULT for matrices: only tensor because of conversion
                origin = "upper")
fig.show()